# Clustering WASI — Cognitive Profiles at 20 Years

Groups participants by **WASI** performance using K-Means (K=2).  
Follows the identical pipeline as `Model1_clustering_GOi.ipynb` — only the input variables change.

| Aspect | Full GO-i | WASI only |
|--------|-----------|----------|
| Variables | 19 (WASI+TAP+CVLT+VMI) | 1 WASI |
| Labelling rule | Higher WASI FSIQ → GO-1 | Higher WASI FSIQ → GO-1 |
| Pipeline | — | Identical |
| MLflow | part_a_clustering | part_a_clustering_wasi |


## 1. Environment Setup & Imports

In [0]:
"""
Suppress threadpoolctl warnings from Databricks multi-threaded environment.
Must be set before any sklearn/numpy imports.
"""
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import io
import logging
import warnings
warnings.filterwarnings("ignore")

# Suppress stderr during sklearn import (Databricks verbosity)
_stderr_orig = sys.stderr
sys.stderr = open(os.devnull, "w")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
)
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests
from collections import Counter
from src.data.load_data import load_data

sys.stderr.close()
sys.stderr = _stderr_orig

print("Imports OK ✓")


## 2. Configuration — WASI Variables

In [0]:
INPUT_FILE   = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
OUT_CLUSTERS = "clusters_GO_WASI_v1.csv"
OUT_CENT_RAW = "centroides_GO_WASI_raw.csv"
OUT_CENT_Z   = "centroides_GO_WASI_zscore.csv"

K_RANGE      = range(2, 9)
K_FINAL      = 2
RANDOM_STATE = 42
N_INIT       = 50
MAX_ITER     = 500
N_RUNS       = 10

IQR_MULT  = 3.0
WINSORIZE = True

GROUP_NAMES  = {1: "KMC", 2: "TC", 3: "Reference"}
COLORS_GO    = {1: "#2E86AB", 2: "#C73E1D"}
COLORS_GROUP = {1: "#2E86AB", 2: "#A23B72", 3: "#F18F01"}

MLFLOW_EXPERIMENT = "/Users/your_user@email.com/saving_brains/part_a_clustering_wasi"

VARS_CLUSTER = [
    "WASI_FullScale4compositescore",
]

ABBREV = {
    "WASI_FullScale4compositescore": "WASI FSIQ",
}

ALL_VARS = VARS_CLUSTER

EXCLUDE_WINSOR = []
VARS_LOG       = []

print("Configuration loaded — WASI clustering variant")
print(f"Variables: {len(VARS_CLUSTER)}")


## 3. Logging & Utility Helpers

In [0]:
# Configure logging to stdout (Databricks captures it in the cell output)
_fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")
_ch  = logging.StreamHandler(sys.stdout)
_ch.setFormatter(_fmt)
logging.basicConfig(level=logging.INFO, handlers=[_ch], force=True)
logger = logging.getLogger("clustering_goi")


def to_num(series: pd.Series) -> pd.Series:
    """Coerce a Series to numeric, turning non-parseable values into NaN."""
    return pd.to_numeric(series, errors="coerce")


def section(title: str = "", width: int = 72) -> None:
    """Log a section header separator."""
    t = title.strip()
    if t:
        separator = "─" * max(2, width - len(t) - 6)
        logger.info(f"\n {t} {separator}")
    else:
        logger.info("─" * width)


print("Logging configured ✓")

## 4. Step 0 — Data Loading & Preprocessing

In [0]:
def load_and_preprocess(path: str):
    """
    Load dataset and prepare the feature matrix for clustering.

    Steps
    -----
    1. Load CSV and validate that expected variables exist.
    2. Verify missingness pattern (MCAR chi-square test per module).
    3. Impute missing values with MICE (IterativeImputer + BayesianRidge).
    4. Standardise features to Z-score (mean=0, SD=1).

    Returns
    -------
    df       : raw DataFrame
    X_imp    : imputed feature matrix (original scale)
    X_sc     : standardised feature matrix
    scaler   : fitted StandardScaler (needed to inverse-transform later)
    vars_ok  : list of variables found in the dataset
    groups   : Series with group membership (1=KMC, 2=TC, 3=Reference)
    """
    section("STEP 0 — DATA LOADING & PREPROCESSING")

    #df = pd.read_csv(path, low_memory=False)
    df = load_data('processed')
    logger.info(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} cols")

    # Identify available variables
    missing_vars = [v for v in ALL_VARS if v not in df.columns]
    if missing_vars:
        logger.warning(f"{len(missing_vars)} expected variables not found: {missing_vars}")
    vars_ok = [v for v in ALL_VARS if v in df.columns]
    logger.info(
        f"Clustering variables: {len(vars_ok)} — WASI domain only"
    )

    # MCAR test: chi-square between missingness indicator and group membership
    section("Missingness audit — MCAR test by module")
    groups_raw = to_num(df["ubicac"]) if "ubicac" in df.columns else pd.Series()
    logger.info("  Module    N     Mean miss    Max miss           MCAR?")
    logger.info("  " + "─" * 54)
    for mod, lst in [("WASI", VARS_CLUSTER)]:
        ok   = [v for v in lst if v in df.columns]
        pcts = df[ok].apply(to_num).isna().mean() * 100
        p_values = []
        for col in ok:
            s = to_num(df[col])
            present = s.notna().astype(int)
            sub = pd.DataFrame({"g": groups_raw, "t": present})
            sub = sub[sub["g"].isin([1, 2, 3])]
            ct  = pd.crosstab(sub["g"], sub["t"])
            if ct.shape == (3, 2):
                try:
                    _, p, _, _ = chi2_contingency(ct)
                    p_values.append(p)
                except Exception:
                    pass
        mcar_label = "MCAR ✓" if not p_values or min(p_values) >= 0.05 \
                     else f"MAR ⚠  p={min(p_values):.3f}"
        logger.info(
            f"  {mod:<8} {len(ok):>4}  {pcts.mean():>9.1f}%  "
            f"{pcts.max():>9.1f}%  {mcar_label}"
        )

    # Build raw feature matrix
    X_raw    = df[vars_ok].apply(to_num)
    nan_total = X_raw.isna().sum().sum()
    n_with_nan = X_raw.isna().any(axis=1).sum()
    logger.info(
        f"Participants with ≥1 NaN: {n_with_nan} ({n_with_nan/len(df)*100:.1f}%) — "
        f"total NaN cells: {nan_total}"
    )

    # MICE imputation
    # Rationale: median imputation collapses SD to 0; MICE preserves variance.
    # Max missingness is 3.7% → imputation bias is minimal.
    section("MICE imputation (IterativeImputer + BayesianRidge)")
    mice = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    X_imp = pd.DataFrame(
        mice.fit_transform(X_raw.values),
        columns=vars_ok,
        index=X_raw.index,
    )
    logger.info(
        f"NaN before MICE: {nan_total} → after: {X_imp.isna().sum().sum()} | "
        f"n preserved: {len(X_imp)}"
    )

    # Imputation quality check: compare mean/SD of imputed vs observed values
    for col in VARS_CLUSTER[:min(2, len(VARS_CLUSTER))]:
        if col not in vars_ok:
            continue
        nan_mask = X_raw[col].isna()
        if nan_mask.sum() == 0:
            continue
        orig = X_raw[col].dropna()
        imp  = X_imp.loc[nan_mask, col]
        logger.info(
            f"  {col:<35} observed: {orig.mean():.1f}±{orig.std():.1f} | "
            f"imputed: {imp.mean():.1f}±{imp.std():.1f}"
        )

    # Z-score standardisation
    scaler = StandardScaler()
    X_sc   = pd.DataFrame(
        scaler.fit_transform(X_imp),
        columns=vars_ok,
        index=X_raw.index,
    )
    logger.info("Z-score standardisation applied (mean≈0, SD≈1) ✓")

    # Group distribution
    groups = to_num(df["ubicac"]) if "ubicac" in df.columns else pd.Series()
    section("Group distribution")
    for g, name in GROUP_NAMES.items():
        logger.info(f"  {name:<12}: n={(groups == g).sum()}")

    return df, X_imp, X_sc, scaler, vars_ok, groups


df, X_imp, X_sc, scaler, vars_ok, groups = load_and_preprocess(INPUT_FILE)


## 5. Step 0b — Outlier Audit & Treatment

In [0]:
# EXCLUDE_WINSOR defined in Section 2 for WASI

# VARS_LOG defined in Section 2 for WASI


def _audit_variable(series: pd.Series, name: str):
    """
    Detect outliers using the IQR method and optionally Winsorize.

    Falls back to P2.5–P97.5 range when IQR=0 (near-constant variable).
    Returns (summary_dict, cleaned_series).
    """
    n_valid = series.notna().sum()
    if n_valid < 10:
        return None, series

    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr    = q3 - q1
    method = "IQR"

    if iqr == 0:
        p025, p975 = series.quantile(0.025), series.quantile(0.975)
        span = p975 - p025
        if span == 0:
            return {
                "variable": name, "n_valid": int(n_valid),
                "IQR": 0.0, "n_outliers": 0, "pct_outliers": 0.0,
                "alert": "N/A", "method": "IQR=0_constant",
                "winsorized": False,
                "note": "IQR=0: near-constant variable. No treatment applied.",
            }, series
        lo, hi = p025 - IQR_MULT * span * 0.5, p975 + IQR_MULT * span * 0.5
        method = "P2.5-P97.5"
        logger.warning(f"  ⚠ {name}: IQR=0 — falling back to P2.5-P97.5")
    else:
        lo, hi = q1 - IQR_MULT * iqr, q3 + IQR_MULT * iqr

    outlier_mask = ((series < lo) | (series > hi)) & series.notna()
    n_out = int(outlier_mask.sum())
    pct   = n_out / n_valid * 100

    series_clean = series.copy()
    if WINSORIZE and n_out > 0:
        series_clean = series_clean.clip(lower=lo, upper=hi)

    alert = "HIGH" if pct > 5 else "MEDIUM" if pct > 1 else "OK"
    return {
        "variable": name, "n_valid": int(n_valid),
        "mean": round(float(series.mean()), 3),
        "sd":   round(float(series.std()), 3),
        "Q1": round(float(q1), 3), "Q3": round(float(q3), 3),
        "IQR": round(float(iqr), 3),
        "lower_bound": round(float(lo), 3),
        "upper_bound": round(float(hi), 3),
        "min_obs": round(float(series.min()), 3),
        "max_obs": round(float(series.max()), 3),
        "n_outliers": n_out, "pct_outliers": round(pct, 2),
        "alert": alert, "method": method,
        "winsorized": WINSORIZE and n_out > 0,
        "note": "" if method == "IQR" else f"IQR=0: used {method}",
    }, series_clean


def audit_outliers(df: pd.DataFrame, vars_ok: list):
    """
    Audit and treat outliers for all clustering variables.

    Returns
    -------
    df_clean   : DataFrame with Winsorization and log(x+1) applied
    df_report  : audit summary as a DataFrame (one row per variable)
    """
    section(f"STEP 0b — OUTLIER AUDIT (±{IQR_MULT}×IQR Winsorization)")

    module_map = [(var, "WASI") for var in VARS_CLUSTER]

    records    = []
    df_clean   = df.copy()
    winsorized = []

    # Extract string literals to avoid f-string nesting issues
    col_mod = "Mod"
    col_var = "Variable"
    col_iqr = "IQR"
    col_nout = "n_out"
    col_pct = "pct"
    col_bounds = "Bounds"
    col_alert = "Alert"
    dash_3 = "---"
    
    logger.info(
        f"  {col_mod:<6} {col_var:<35} {col_iqr:>8} {col_nout:>6} "
        f"{col_pct:>6}  {col_bounds:>28}  {col_alert}"
    )
    logger.info("  " + "─" * 100)

    for col, mod in module_map:
        if col not in vars_ok or col not in df.columns:
            continue

        if col in EXCLUDE_WINSOR:
            skip_message = "IQR=0: ceiling effect — skipped"
            logger.info(
                f"  {mod:<6} {col:<35} {dash_3:>8} {dash_3:>6} "
                f"{dash_3:>6}  {skip_message:>28}  N/A"
            )
            records.append({
                "module": mod, "variable": col, "n_outliers": 0,
                "pct_outliers": 0, "alert": "N/A", "winsorized": False,
                "note": "IQR=0 / ceiling effect — no Winsorization",
            })
            continue

        res, s_clean = _audit_variable(to_num(df[col]), col)
        if res is None:
            continue
        res["module"] = mod
        records.append(res)

        icon = "🔴" if res["alert"] == "HIGH" \
               else "🟡" if res["alert"] == "MEDIUM" else "✓"
        if res["n_outliers"] > 0:
            logger.info(
                f"  {icon} {mod:<5} {col:<35} {res['IQR']:>8.2f} "
                f"{res['n_outliers']:>6} {res['pct_outliers']:>5.1f}%  "
                f"[{res['lower_bound']:>12.2f},{res['upper_bound']:>12.2f}]  "
                f"{res['alert']}"
            )
        if WINSORIZE and res["n_outliers"] > 0:
            df_clean[col] = s_clean
            winsorized.append(col)

    # log(x+1) for right-skewed CVLT error counts
    section("log(x+1) transform — right-skewed CVLT variables")
    log_applied = []
    for col in VARS_LOG:
        if col not in vars_ok or col not in df.columns:
            continue
        s_orig = to_num(df[col])
        if (s_orig.dropna() < 0).any():
            logger.warning(f"  ⚠ {col}: negative values found — log transform skipped")
            continue
        s_log = np.log1p(s_orig)
        skew_before = round(float(s_orig.dropna().skew()), 3)
        skew_after  = round(float(s_log.dropna().skew()), 3)
        logger.info(f"  {col:<35} skew: {skew_before:>7.3f} → {skew_after:>7.3f}  ✓")
        df_clean[col] = s_log
        log_applied.append(col)
        for r in records:
            if r["variable"] == col:
                r["note"] = f"log(x+1) applied. skew: {skew_before}→{skew_after}"

    winsor_msg = "none" if not winsorized else winsorized
    log_msg = "none" if not log_applied else log_applied
    logger.info(f"Winsorized: {winsor_msg}")
    logger.info(f"Log-transformed: {log_msg}")
    logger.info("Unchanged (ceiling effect): CVLT_Hits")

    df_report = pd.DataFrame(records)
    return df_clean, df_report


def plot_outlier_comparison(df_orig, df_clean, vars_ok, df_report):
    """Display side-by-side boxplots for variables that were modified."""
    changed = df_report[
        (df_report["n_outliers"] > 0) |
        (df_report["note"].str.contains("log", na=False))
    ]["variable"].tolist()
    changed = [v for v in changed if v in vars_ok]

    if not changed:
        print("No variables were modified — comparison plot skipped.")
        return

    n_cols = min(4, len(changed))
    n_rows = (len(changed) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes_flat = np.array(axes).flatten()

    for i, col in enumerate(changed):
        ax = axes_flat[i]
        s_orig  = to_num(df_orig[col]).dropna()
        s_clean = to_num(df_clean[col]).dropna()
        bp = ax.boxplot(
            [s_orig.values, s_clean.values],
            patch_artist=True, widths=0.5,
            medianprops=dict(color="white", linewidth=2),
        )
        bp["boxes"][0].set_facecolor("#AAAAAA")
        bp["boxes"][0].set_alpha(0.7)
        bp["boxes"][1].set_facecolor("#2E86AB")
        bp["boxes"][1].set_alpha(0.85)

        row_info = df_report[df_report["variable"] == col]
        note = ""
        if not row_info.empty:
            r = row_info.iloc[0]
            if r["n_outliers"] > 0:
                note = f"n_out={r['n_outliers']} ({r['pct_outliers']:.1f}%) — Winsorized"
            elif "log" in str(r.get("note", "")):
                note = "log(x+1) applied"

        short = col.replace("CVLT_","").replace("TAP_","").replace("VMI_","").replace("WASI_","")
        ax.set_xticklabels(["Original", "Cleaned"], fontsize=8)
        ax.set_title(short, fontweight="bold", fontsize=9)
        if note:
            ax.set_xlabel(note, fontsize=7.5, color="#C73E1D")

    for j in range(len(changed), len(axes_flat)):
        fig.delaxes(axes_flat[j])

    title_line1 = "Clustering Variables — Original vs Cleaned"
    title_line2 = f"Winsorization ±{IQR_MULT}×IQR  |  log(x+1) for skewed CVLT"
    title_line3 = "Grey=original  |  Blue=cleaned"
    plt.suptitle(
        f"{title_line1}\n{title_line2}\n{title_line3}",
        fontweight="bold", fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.show()


df_clean, df_report_outliers = audit_outliers(df, vars_ok)
plot_outlier_comparison(df, df_clean, vars_ok, df_report_outliers)

print("\n✓ Clean dataset ready for MICE. Use df_clean for subsequent steps.")

## 6. Step 1 — Optimal K Selection

In [0]:
def select_optimal_k(X_sc: pd.DataFrame):
    """
    Evaluate K-Means for K in K_RANGE using Elbow, Silhouette, and
    Calinski-Harabasz metrics. Logs results and displays plots inline.

    Returns
    -------
    silhouette_at_k_final : float — silhouette score at the selected K
    """
    section("STEP 1 — OPTIMAL K SELECTION")

    X = X_sc.values
    inertias, silhouettes, ch_scores, db_scores = [], [], [], []

    # Extract string literals to avoid f-string nesting issues
    col_k = "K"
    col_inertia = "Inertia"
    col_sil = "Silhouette"
    col_ch = "Calinski-H"
    col_db = "Davies-B"
    
    logger.info(
        f"  {col_k:>3}  {col_inertia:>10}  {col_sil:>12}  "
        f"{col_ch:>12}  {col_db:>10}"
    )
    logger.info("  " + "─" * 55)

    for k in K_RANGE:
        km  = KMeans(n_clusters=k, init="k-means++", n_init=N_INIT,
                     max_iter=MAX_ITER, random_state=RANDOM_STATE)
        lbl = km.fit_predict(X)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X, lbl))
        ch_scores.append(calinski_harabasz_score(X, lbl))
        db_scores.append(davies_bouldin_score(X, lbl))
        tag = " ← selected" if k == K_FINAL else ""
        logger.info(
            f"  {k:>3}  {km.inertia_:>10.1f}  {silhouettes[-1]:>12.4f}  "
            f"{ch_scores[-1]:>12.2f}  {db_scores[-1]:>10.4f}{tag}"
        )

    # Inline figure — three metric panels
    ks  = list(K_RANGE)
    idx = ks.index(K_FINAL)
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].plot(ks, inertias, "o-", color="steelblue", lw=2, ms=7)
    axes[0].axvline(K_FINAL, color="red", ls="--", alpha=0.7, label=f"K={K_FINAL}")
    axes[0].set_title("Elbow Method (Inertia)", fontweight="bold")
    axes[0].set_xlabel("K")
    axes[0].set_ylabel("Inertia (WCSS)")
    axes[0].legend()

    axes[1].plot(ks, silhouettes, "o-", color="#2E86AB", lw=2, ms=7)
    axes[1].axvline(K_FINAL, color="red", ls="--", alpha=0.7, label=f"K={K_FINAL}")
    axes[1].annotate(
        f"{silhouettes[idx]:.3f}",
        xy=(K_FINAL, silhouettes[idx]),
        xytext=(K_FINAL + 0.3, silhouettes[idx] + 0.005),
        fontsize=10, color="red", fontweight="bold",
    )
    axes[1].set_title("Silhouette Score", fontweight="bold")
    axes[1].set_xlabel("K")
    axes[1].set_ylabel("Silhouette")
    axes[1].legend()

    axes[2].plot(ks, ch_scores, "o-", color="#A23B72", lw=2, ms=7)
    axes[2].axvline(K_FINAL, color="red", ls="--", alpha=0.7, label=f"K={K_FINAL}")
    axes[2].set_title("Calinski-Harabasz Score", fontweight="bold")
    axes[2].set_xlabel("K")
    axes[2].set_ylabel("CH (higher = better)")
    axes[2].legend()

    plt.suptitle(
        f"Optimal K Selection — WASI Clustering v1\nSelected K={K_FINAL}",
        fontweight="bold", fontsize=13,
    )
    plt.tight_layout()
    plt.show()

    return silhouettes[idx]


sil_at_k = select_optimal_k(X_sc)

## 7. Step 2 — K-Means Clustering

In [0]:
def fit_kmeans(X_sc: pd.DataFrame):
    """
    Fit K-Means with K_FINAL clusters and assess solution stability.

    Stability is measured by re-running K-Means with different random seeds
    and comparing Silhouette scores. SD < 0.01 indicates a stable solution.

    Returns
    -------
    km         : fitted KMeans model
    labels     : cluster assignment array
    sil_global : overall Silhouette score
    db         : Davies-Bouldin score
    """
    section(f"STEP 2 — K-MEANS (K={K_FINAL})")
    X = X_sc.values

    km     = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=N_INIT,
                    max_iter=MAX_ITER, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)

    sil_global = silhouette_score(X, labels)
    ch         = calinski_harabasz_score(X, labels)
    db         = davies_bouldin_score(X, labels)

    logger.info(f"Inertia (WCSS)      : {km.inertia_:.2f}")
    logger.info(f"Silhouette (global) : {sil_global:.4f}  (>0.25 acceptable | >0.50 good)")
    logger.info(f"Calinski-Harabasz   : {ch:.2f}  (higher = better)")
    logger.info(f"Davies-Bouldin      : {db:.4f}  (lower = better)")

    # Cluster size distribution
    section("Cluster size distribution (pre-labelling)")
    for c, n in sorted(Counter(labels).items()):
        logger.info(f"  Cluster {c}: n={n} ({n/len(labels)*100:.1f}%)")

    # Stability across 10 independent runs
    section("Stability validation (10 independent runs)")
    
    # Extract string literals to avoid f-string nesting issues
    col_run = "Run"
    col_sil = "Silhouette"
    col_inertia = "Inertia"
    
    logger.info(f"  {col_run:>4}  {col_sil:>12}  {col_inertia:>12}")
    logger.info("  " + "─" * 32)
    sil_runs = []
    for run in range(N_RUNS):
        km_r  = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=10,
                       max_iter=MAX_ITER, random_state=run * 7)
        lbl_r = km_r.fit_predict(X)
        sil_r = silhouette_score(X, lbl_r)
        sil_runs.append(sil_r)
        logger.info(f"  {run+1:>4}  {sil_r:>12.4f}  {km_r.inertia_:>12.1f}")

    stability_msg = "Stable solution (SD < 0.01) ✓" if np.std(sil_runs) < 0.01 \
                    else "Moderate variance — consider reviewing K"
    logger.info(
        f"\n  Mean={np.mean(sil_runs):.4f}  SD={np.std(sil_runs):.4f}  "
        f"Min={np.min(sil_runs):.4f}  Max={np.max(sil_runs):.4f}\n  {stability_msg}"
    )

    return km, labels, sil_global, db


km, labels, sil_global, db_score = fit_kmeans(X_sc)

## 8. Step 3 — Labelling & Characterisation

In [0]:
def label_and_characterise(df, X_imp, X_sc, labels, vars_ok, groups):
    """
    Assign GO-i labels and characterise group differences.

    Labelling rule (TAP): Lower RT = better performance.
    Statistical test: Mann-Whitney U (two-sided) per variable.
    """
    section("STEP 3 — GO-i LABELLING & CHARACTERISATION")

    FSIQ_COL  = "WASI_FullScale4compositescore"
    df_tmp    = X_imp.copy()
    df_tmp["_cluster"] = labels

    # Higher FSIQ = better performance -> cluster with HIGHER mean FSIQ -> GO-1
    fsiq_means = {c: df_tmp.loc[df_tmp["_cluster"] == c, FSIQ_COL].mean()
                  for c in range(K_FINAL)}
    ordered   = sorted(fsiq_means, key=lambda c: fsiq_means[c], reverse=True)
    remap     = {ordered[i]: i + 1 for i in range(K_FINAL)}
    labels_go = np.array([remap[l] for l in labels])
    LABEL_GO  = {1: "GO-1 High IQ", 2: "GO-2 Low IQ"}

    section("3A — Variable means by GO-i (Mann-Whitney U)")
    header = "".join([f"{LABEL_GO[g]:>18}" for g in [1, 2]])
    col_var = "Variable"
    logger.info(f"  {col_var:<30} {header}  p-value")
    logger.info("  " + "─" * 78)

    X_labelled = X_imp.copy()
    X_labelled["GO_i"] = labels_go
    p_values = {}

    for col in vars_ok:
        v1 = X_labelled.loc[X_labelled["GO_i"] == 1, col].dropna().values
        v2 = X_labelled.loc[X_labelled["GO_i"] == 2, col].dropna().values
        try:
            _, p = mannwhitneyu(v1, v2, alternative="two-sided")
        except Exception:
            p = np.nan
        p_values[col] = p
        sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        means = "".join([f"{np.mean(v):>18.2f}" for v in [v1, v2]])
        logger.info(f"  {ABBREV.get(col, col):<30} {means}  {p:.6f} {sig}")

    section("3B — Group composition by GO-i (KMC / TC / Reference)")
    col_goi   = "GO-i"
    col_kmc   = "KMC"
    col_tc    = "TC"
    col_ref   = "Ref"
    col_total = "Total"
    col_pkmc  = "%KMC"
    col_ptc   = "%TC"
    col_pref  = "%Ref"
    logger.info(
        f"  {col_goi:<12} {col_kmc:>8} {col_tc:>8} {col_ref:>8} "
        f"{col_total:>8}  {col_pkmc:>6} {col_ptc:>6} {col_pref:>6}"
    )
    logger.info("  " + "─" * 72)
    for go in [1, 2]:
        mask  = labels_go == go
        n_tot = mask.sum()
        n_kmc = (mask & (groups == 1)).sum()
        n_tc  = (mask & (groups == 2)).sum()
        n_ref = (mask & (groups == 3)).sum()
        logger.info(
            f"  {LABEL_GO[go]:<12} {n_kmc:>8} {n_tc:>8} {n_ref:>8} "
            f"{n_tot:>8}  {n_kmc/n_tot*100:>5.1f}% "
            f"{n_tc/n_tot*100:>5.1f}% {n_ref/n_tot*100:>5.1f}%"
        )

    section("3C — Executive summary (WASI variant)")
    col_goi2  = "GO-i"
    col_n     = "n"
    col_pct   = "%"
    col_fsiq  = "FSIQ"
    logger.info(
        f"  {col_goi2:<12} {col_n:>5}  {col_pct:>5}  {col_fsiq:>10}"
    )
    logger.info("  " + "─" * 42)
    FSIQ_COL2 = "WASI_FullScale4compositescore"
    for go in [1, 2]:
        mask = labels_go == go
        n    = mask.sum()
        fsiq = X_labelled.loc[mask, FSIQ_COL2].mean() if FSIQ_COL2 in X_labelled.columns else np.nan
        logger.info(
            f"  {LABEL_GO[go]:<12} {n:>5}  {n/len(labels_go)*100:>4.1f}%  {fsiq:>10.1f}"
        )

    return labels_go, X_labelled, LABEL_GO, p_values


labels_go, X_labelled, LABEL_GO, p_values = label_and_characterise(
    df, X_imp, X_sc, labels, vars_ok, groups
)


## 9. Step 4 — Characterisation Figures

In [0]:
def plot_characterisation(X_imp, X_sc, labels_go, groups, vars_ok, LABEL_GO):
    """
    Generate six inline visualisations for GO-i cluster characterisation.

    Figures
    -------
    Fig 1 : Centroid heatmap (Z-score)
    Fig 2 : PCA 2D scatter — GO-i labels vs group labels (skipped if <2 features)
    Fig 3 : Group composition bar charts
    Fig 4 : Silhouette diagram
    Fig 5 : Radar chart by cognitive domain
    Fig 6 : PCA 3D scatter (skipped if <3 features)

    Returns
    -------
    sil_samples : ndarray — per-sample silhouette scores
    sil_global  : float
    fig_objects : list of matplotlib Figure objects (used by MLflow logger)
    """
    section("STEP 4 — CHARACTERISATION FIGURES")
    X    = X_sc.values
    abr  = [ABBREV.get(v, v) for v in vars_ok]
    figs = []
    n_features = len(vars_ok)

    # ── Fig 1: Centroid heatmap ────────────────────────────────────────
    centroids_z = np.array([X_sc[labels_go == g].mean().values for g in [1, 2]])
    df_heat = pd.DataFrame(
        centroids_z,
        index=[f"{LABEL_GO[g]}\\n(n={int((labels_go==g).sum())})" for g in [1, 2]],
        columns=abr,
    )
    fig1, ax = plt.subplots(figsize=(16, 4))
    sns.heatmap(
        df_heat, cmap="RdBu_r", center=0, vmin=-1.5, vmax=1.5,
        annot=True, fmt=".2f", linewidths=0.4, ax=ax,
        cbar_kws={"label": "Z-score", "shrink": 0.6},
        annot_kws={"size": 7.5},
    )
    ax.set_title(
        f"Centroid Profiles — WASI (Z-score)\\nK={K_FINAL} | {len(vars_ok)} variables",
        fontweight="bold", fontsize=12,
    )
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=10)
    plt.tight_layout()
    plt.show()
    figs.append(fig1)

    # ── Fig 2: PCA 2D (conditional) ────────────────────────────────────
    if n_features >= 2:
        pca      = PCA(n_components=2, random_state=RANDOM_STATE)
        X_pca    = pca.fit_transform(X)
        var_exp  = pca.explained_variance_ratio_ * 100
        cent_pca = pca.transform(centroids_z)

        fig2, axes = plt.subplots(1, 2, figsize=(14, 6))
        for go in [1, 2]:
            mask = labels_go == go
            axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                            c=COLORS_GO[go], label=LABEL_GO[go],
                            alpha=0.55, s=22, edgecolors="none")
        for go in [1, 2]:
            axes[0].scatter(cent_pca[go-1, 0], cent_pca[go-1, 1],
                            c=COLORS_GO[go], marker="*", s=350,
                            edgecolors="black", lw=1.2, zorder=5)
        axes[0].set_xlabel(f"PC1 ({var_exp[0]:.1f}% var)", fontsize=11)
        axes[0].set_ylabel(f"PC2 ({var_exp[1]:.1f}% var)", fontsize=11)
        axes[0].set_title("PCA — GO-i Clusters\\n(★ = centroid)", fontweight="bold")
        axes[0].legend(fontsize=10, markerscale=1.8)

        for g, name in GROUP_NAMES.items():
            mask = groups == g
            axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                            c=COLORS_GROUP[g], label=name,
                            alpha=0.55, s=22, edgecolors="none")
        axes[1].set_xlabel(f"PC1 ({var_exp[0]:.1f}% var)", fontsize=11)
        axes[1].set_ylabel(f"PC2 ({var_exp[1]:.1f}% var)", fontsize=11)
        axes[1].set_title("PCA — KMC / TC / Reference groups", fontweight="bold")
        axes[1].legend(fontsize=10, markerscale=1.8)
        plt.suptitle(f"PCA Projection — WASI Clustering (K={K_FINAL})", fontweight="bold", fontsize=13)
        plt.tight_layout()
        plt.show()
        figs.append(fig2)
    else:
        logger.info(f"  Skipping PCA 2D (requires ≥2 features, found {n_features})")

    # ── Fig 3: Group composition ───────────────────────────────────────
    fig3, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(K_FINAL)
    w = 0.25
    for gi, (g, name, color) in enumerate(
        zip([1, 2, 3], list(GROUP_NAMES.values()), list(COLORS_GROUP.values()))
    ):
        vals = [
            ((labels_go == go) & (groups == g)).sum() / (labels_go == go).sum() * 100
            for go in [1, 2]
        ]
        axes[0].bar(x + gi * w, vals, w, label=name, color=color, alpha=0.85, edgecolor="white")
    axes[0].set_xticks(x + w)
    axes[0].set_xticklabels(
        [f"{LABEL_GO[go]}\\n(n={(labels_go==go).sum()})" for go in [1, 2]], fontsize=10
    )
    axes[0].set_ylabel("% within cluster", fontsize=11)
    axes[0].set_title("Group distribution within GO-i", fontweight="bold")
    axes[0].legend(fontsize=10)
    axes[0].axhline(33.3, color="gray", ls="--", alpha=0.5, lw=1.2)

    for gi, (g, name, color) in enumerate(
        zip([1, 2, 3], list(GROUP_NAMES.values()), list(COLORS_GROUP.values()))
    ):
        n_g = (groups == g).sum()
        bottom = 0
        for go, go_col in zip([1, 2], list(COLORS_GO.values())):
            pct = ((labels_go == go) & (groups == g)).sum() / n_g * 100
            axes[1].bar(gi, pct, bottom=bottom, color=go_col,
                        alpha=0.85, edgecolor="white", width=0.6)
            if pct > 5:
                axes[1].text(gi, bottom + pct / 2, f"{pct:.0f}%",
                             ha="center", va="center", fontsize=9,
                             color="white", fontweight="bold")
            bottom += pct
    axes[1].set_xticks(range(K_FINAL))
    axes[1].set_xticklabels(
        [f"{GROUP_NAMES[g]}\\n(n={(groups==g).sum()})" for g in [1, 2]], fontsize=10
    )
    axes[1].set_ylabel("% within group", fontsize=11)
    axes[1].set_title("GO-i distribution within groups", fontweight="bold")
    handles = [mpatches.Patch(color=COLORS_GO[g], label=LABEL_GO[g]) for g in [1, 2]]
    axes[1].legend(handles=handles, fontsize=9, loc="lower right")
    plt.suptitle("Group × GO-i Composition", fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.show()
    figs.append(fig3)

    # ── Fig 4: Silhouette diagram ──────────────────────────────────────
    sil_samples = silhouette_samples(X, labels_go)
    sil_global  = sil_samples.mean()

    fig4, ax = plt.subplots(figsize=(8, 6))
    y_lower = 10
    for go in [1, 2]:
        sil_go = np.sort(sil_samples[labels_go == go])
        n_go   = sil_go.shape[0]
        ax.fill_betweenx(np.arange(y_lower, y_lower + n_go),
                         0, sil_go, alpha=0.75, color=COLORS_GO[go],
                         label=f"{LABEL_GO[go]} (n={n_go})")
        ax.text(-0.04, y_lower + n_go / 2, LABEL_GO[go], fontsize=8.5, va="center")
        y_lower += n_go + 10
    ax.axvline(sil_global, color="red", ls="--", lw=1.8, label=f"Mean = {sil_global:.3f}")
    ax.set_xlabel("Silhouette Score", fontsize=11)
    ax.set_ylabel("Participants per cluster", fontsize=11)
    ax.set_title(
        "Silhouette Diagram by GO-i\\n(width = individual assignment quality)",
        fontweight="bold", fontsize=12,
    )
    ax.legend(fontsize=9, loc="lower right")
    plt.tight_layout()
    plt.show()
    figs.append(fig4)

    # ── Fig 5: Radar by domain ─────────────────────────────────────────
    cent_raw  = np.array([X_imp[labels_go == g].mean().values for g in [1, 2]])
    cent_norm = (cent_raw - cent_raw.min(axis=0)) / \
                (cent_raw.max(axis=0) - cent_raw.min(axis=0) + 1e-9)

    # Single-domain: one spoke per variable
    dom_vals = [
        [cent_norm[go - 1, vars_ok.index(var)] for var in vars_ok]
        for go in [1, 2]
    ]
    dom_labels = [ABBREV.get(var, var) for var in vars_ok]
    N      = len(dom_labels)
    angles = [n / N * 2 * np.pi for n in range(N)] + [0]

    fig5, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    for go in [1, 2]:
        vals = dom_vals[go - 1] + dom_vals[go - 1][:1]
        ax.plot(angles, vals, "o-", color=COLORS_GO[go], lw=2, label=LABEL_GO[go])
        ax.fill(angles, vals, color=COLORS_GO[go], alpha=0.12)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dom_labels, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(["25%", "50%", "75%"], fontsize=8, color="gray")
    ax.set_title(
        "Cognitive Profile by Domain — GO-i\\n(normalised 0–1 per variable)",
        fontweight="bold", fontsize=12, pad=20,
    )
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
    plt.tight_layout()
    plt.show()
    figs.append(fig5)

    # ── Fig 6: PCA 3D (conditional) ────────────────────────────────────
    if n_features >= 3:
        pca3     = PCA(n_components=3, random_state=RANDOM_STATE)
        X_pca3   = pca3.fit_transform(X)
        var_exp3 = pca3.explained_variance_ratio_ * 100
        cent_3d  = np.array([X_pca3[labels_go == g].mean(axis=0) for g in [1, 2]])

        fig6 = plt.figure(figsize=(16, 7))
        ax1  = fig6.add_subplot(121, projection="3d")
        for go in [1, 2]:
            mask = labels_go == go
            ax1.scatter(X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2],
                        c=COLORS_GO[go], label=LABEL_GO[go],
                        alpha=0.45, s=18, edgecolors="none", depthshade=True)
        for go in [1, 2]:
            ax1.scatter(cent_3d[go-1, 0], cent_3d[go-1, 1], cent_3d[go-1, 2],
                        c=COLORS_GO[go], marker="*", s=400,
                        edgecolors="black", linewidths=0.8, zorder=10, depthshade=False)
            ax1.text(cent_3d[go-1, 0], cent_3d[go-1, 1], cent_3d[go-1, 2] + 0.15,
                     f"GO-{go}", fontsize=8.5, fontweight="bold",
                     color=COLORS_GO[go], ha="center")
        ax1.set_xlabel(f"PC1 ({var_exp3[0]:.1f}%)", fontsize=9, labelpad=6)
        ax1.set_ylabel(f"PC2 ({var_exp3[1]:.1f}%)", fontsize=9, labelpad=6)
        ax1.set_zlabel(f"PC3 ({var_exp3[2]:.1f}%)", fontsize=9, labelpad=6)
        ax1.set_title(
            f"PCA 3D — GO-i Clusters\\nVariance: {sum(var_exp3):.1f}%  (★=centroid)",
            fontweight="bold", fontsize=11,
        )
        ax1.legend(fontsize=9, loc="upper left", markerscale=1.5)
        ax1.view_init(elev=20, azim=45)

        ax2 = fig6.add_subplot(122, projection="3d")
        for g, name in GROUP_NAMES.items():
            mask = groups == g
            ax2.scatter(X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2],
                        c=COLORS_GROUP[g], label=name,
                        alpha=0.45, s=18, edgecolors="none", depthshade=True)
        ax2.set_xlabel(f"PC1 ({var_exp3[0]:.1f}%)", fontsize=9, labelpad=6)
        ax2.set_ylabel(f"PC2 ({var_exp3[1]:.1f}%)", fontsize=9, labelpad=6)
        ax2.set_zlabel(f"PC3 ({var_exp3[2]:.1f}%)", fontsize=9, labelpad=6)
        ax2.set_title("PCA 3D — KMC / TC / Reference\\n(same projection)", fontweight="bold", fontsize=11)
        ax2.legend(fontsize=9, loc="upper left", markerscale=1.5)
        ax2.view_init(elev=20, azim=45)
        plt.suptitle(
            f"PCA 3D Projection — WASI Clustering (K={K_FINAL})\\n"
            f"PC1+PC2+PC3 = {sum(var_exp3):.1f}% total variance",
            fontweight="bold", fontsize=13, y=1.01,
        )
        plt.tight_layout()
        plt.show()
        figs.append(fig6)
    else:
        logger.info(f"  Skipping PCA 3D (requires ≥3 features, found {n_features})")

    return sil_samples, sil_global, figs


sil_samples, sil_global_fig, fig_objects = plot_characterisation(
    X_imp, X_sc, labels_go, groups, vars_ok, LABEL_GO
)

## 10. Step 5 — Statistical Tests (FDR-corrected)

In [0]:
def run_statistical_tests(X_imp, labels_go, vars_ok, p_values, LABEL_GO):
    """
    Apply Benjamini-Hochberg FDR correction to raw Mann-Whitney p-values.

    Returns
    -------
    q_values : ndarray — FDR-adjusted q-values aligned with vars_ok
    """
    section("STEP 5 — STATISTICAL TESTS (FDR-corrected)")
    logger.info("K=2: Mann-Whitney U (two-sided) per variable.")
    logger.info("Multiple comparison correction: FDR Benjamini-Hochberg.\n")

    X_tmp = X_imp.copy()
    X_tmp["GO_i"] = labels_go

    n_sig_raw = sum(1 for v in vars_ok if p_values.get(v, 1) < 0.05)
    logger.info(f"Variables with p<0.05 (raw): {n_sig_raw}/{len(vars_ok)}")

    # FDR correction
    ps_arr   = np.array([p_values.get(v, np.nan) for v in vars_ok], dtype=float)
    mask_ok  = ~np.isnan(ps_arr)
    q_values = np.full(len(ps_arr), np.nan)
    if mask_ok.sum() > 0:
        _, qs_adj, _, _ = multipletests(ps_arr[mask_ok], method="fdr_bh")
        q_values[mask_ok] = qs_adj

    logger.info(f"Variables with q<0.05 (FDR): {int((q_values < 0.05).sum())}/{len(vars_ok)}")
    logger.info(f"Variables with q<0.10 (FDR): {int((q_values < 0.10).sum())}/{len(vars_ok)}")

    # Full results table sorted by p-value
    section("All variables — GO-1 vs GO-2 (sorted by p-value)")
    
    # Extract string literals to avoid f-string nesting issues
    col_var = "Variable"
    col_go1_mean = "GO-1 mean"
    col_go2_mean = "GO-2 mean"
    col_p_raw = "p_raw"
    col_q_fdr = "q_FDR"
    col_sig = "Sig"
    
    logger.info(
        f"  {col_var:<30} {col_go1_mean:>12} {col_go2_mean:>12} "
        f"{col_p_raw:>10}  {col_q_fdr:>10}  {col_sig}"
    )
    logger.info("  " + "─" * 84)
    for i in np.argsort(ps_arr):
        v, p, q = vars_ok[i], ps_arr[i], q_values[i]
        if np.isnan(p):
            continue
        m1 = X_tmp.loc[X_tmp["GO_i"] == 1, v].mean()
        m2 = X_tmp.loc[X_tmp["GO_i"] == 2, v].mean()
        sig = "***" if q < 0.001 else "**" if q < 0.01 \
              else "*" if q < 0.05 else "†" if q < 0.10 else "ns"
        logger.info(
            f"  {ABBREV.get(v, v):<30} {m1:>12.3f} {m2:>12.3f} "
            f"{p:>10.6f}  {q:>10.6f}  {sig}"
        )

    return q_values


q_values = run_statistical_tests(X_imp, labels_go, vars_ok, p_values, LABEL_GO)

## 11. Step 6 — MLflow Experiment Tracking

In [0]:
def log_to_mlflow(
    km, labels_go, vars_ok, sil_global, db_score,
    sil_samples, fig_objects, q_values, p_values,
):
    """
    Log the complete clustering run to MLflow.

    Logged items
    ------------
    params  : all hyperparameters (K, n_init, imputation strategy, etc.)
    metrics : Silhouette, Davies-Bouldin, cluster sizes, significance counts
    model   : fitted KMeans model (saved as pickle artifact)
    artifacts:
        - clusters_GOiv4.csv  : GO-i assignments per participant
        - centroids_raw.csv   : cluster centroids in original scale
        - centroids_zscore.csv: cluster centroids in Z-score
        - fig_*.png           : all characterisation figures
    """
    section("STEP 6 — MLFLOW LOGGING")

    try:
        mlflow.set_experiment(MLFLOW_EXPERIMENT)

        with mlflow.start_run(run_name=f"clustering_GO_WASI_K2_v1") as run:

            # ── Parameters ────────────────────────────────────────────────
            mlflow.log_params({
                "k_final":          K_FINAL,
                "k_range":          f"{min(K_RANGE)}-{max(K_RANGE)}",
                "n_init":           N_INIT,
                "max_iter":         MAX_ITER,
                "n_stability_runs": N_RUNS,
                "random_state":     RANDOM_STATE,
                "imputation":       "MICE_BayesianRidge_10iter",
                "outlier_method":   f"Winsorization_IQR_{IQR_MULT}x",
                "skewed_vars":      "log1p_CVLT_Intrusions_Repetitions_FP",
                "domain":           "WASI",
                "n_vars":           len(vars_ok),
                "n_features":       len(vars_ok),
                "labelling_rule":   "GO1=higher_WASI_FSIQ_mean",
            })

            # ── Metrics ───────────────────────────────────────────────────
            ps_arr = np.array([p_values.get(v, np.nan) for v in vars_ok], dtype=float)
            mlflow.log_metrics({
                "silhouette_global":        round(float(sil_global), 4),
                "davies_bouldin":           round(float(db_score), 4),
                "silhouette_go1":           round(float(sil_samples[labels_go == 1].mean()), 4),
                "silhouette_go2":           round(float(sil_samples[labels_go == 2].mean()), 4),
                "n_go1":                    int((labels_go == 1).sum()),
                "n_go2":                    int((labels_go == 2).sum()),
                "n_significant_raw_p05":    int((ps_arr < 0.05).sum()),
                "n_significant_fdr_q05":    int((q_values < 0.05).sum()),
                "n_significant_fdr_q10":    int((q_values < 0.10).sum()),
                "inertia":                  round(float(km.inertia_), 2),
            })

            # ── KMeans model & artifacts ─────────────────────────────────────
            import tempfile
            import pickle
            with tempfile.TemporaryDirectory() as tmp:
                # Save KMeans model as pickle
                model_path = os.path.join(tmp, "kmeans_model.pkl")
                with open(model_path, "wb") as f:
                    pickle.dump(km, f)
                mlflow.log_artifact(model_path, artifact_path="model")
                logger.info("KMeans model logged to MLflow as pickle artifact")

                # GO-i assignments
                id_cols = [c for c in ["code", "ubicac", "preterm", "lessthan1801g"]
                           if c in df.columns]
                df_out = df[id_cols].copy()
                df_out["GO_i"]        = labels_go
                df_out["GO_i_label"]  = [LABEL_GO[g] for g in labels_go]
                df_out["sil_sample"]  = sil_samples.round(4)
                clusters_path = os.path.join(tmp, "clusters_GO_WASI_v1.csv")
                df_out.to_csv(clusters_path, index=False)
                mlflow.log_artifact(clusters_path, artifact_path="outputs")

                # Centroids — original scale
                X_tmp = X_imp.copy()
                X_tmp["GO_i"] = labels_go
                df_cent_raw = X_tmp.groupby("GO_i")[vars_ok].mean().round(3)
                df_cent_raw.index = [LABEL_GO[i] for i in df_cent_raw.index]
                cent_raw_path = os.path.join(tmp, "centroids_raw.csv")
                df_cent_raw.to_csv(cent_raw_path)
                mlflow.log_artifact(cent_raw_path, artifact_path="outputs")

                # Centroids — Z-score
                X_sc_tmp = X_sc.copy()
                X_sc_tmp["GO_i"] = labels_go
                df_cent_z = X_sc_tmp.groupby("GO_i")[vars_ok].mean().round(4)
                df_cent_z.index = [LABEL_GO[i] for i in df_cent_z.index]
                cent_z_path = os.path.join(tmp, "centroids_zscore.csv")
                df_cent_z.to_csv(cent_z_path)
                mlflow.log_artifact(cent_z_path, artifact_path="outputs")

                # Figures — save to temp, log, keep display inline
                fig_names = [
                    "fig_01_centroid_heatmap.png",
                    "fig_02_pca_2d.png",
                    "fig_03_group_composition.png",
                    "fig_04_silhouette.png",
                    "fig_05_radar.png",
                    "fig_06_pca_3d.png",
                ]
                for fig, name in zip(fig_objects, fig_names):
                    fig_path = os.path.join(tmp, name)
                    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
                    mlflow.log_artifact(fig_path, artifact_path="figures")

                logger.info("All artifacts logged to MLflow ✓")

            run_id = run.info.run_id
            logger.info(f"\nMLflow run ID  : {run_id}")
            logger.info(f"Experiment     : {MLFLOW_EXPERIMENT}")
            logger.info(
                "\nKey metrics logged:\n"
                f"  silhouette_global = {sil_global:.4f}\n"
                f"  davies_bouldin    = {db_score:.4f}\n"
                f"  n_go1             = {int((labels_go==1).sum())}\n"
                f"  n_go2             = {int((labels_go==2).sum())}\n"
                f"  n_sig_fdr_q05     = {int((q_values<0.05).sum())}"
            )

            return run_id
            
    except Exception as e:
        logger.error(f"MLflow logging failed: {type(e).__name__}: {e}")
        logger.warning("Skipping MLflow logging due to configuration error.")
        logger.info("Results are still available in memory and from previous cells.")
        return None


run_id = log_to_mlflow(
    km, labels_go, vars_ok, sil_global, db_score,
    sil_samples, fig_objects, q_values, p_values,
)

if run_id:
    print(f"\n✓ MLflow run completed. Run ID: {run_id}")
else:
    print("\n⚠️  MLflow logging skipped due to cluster configuration. Run completed successfully.")

## 12. Run Summary

In [0]:
section("RUN SUMMARY — Clustering WASI v1")
logger.info(f"  Participants           : {len(labels_go)}")
logger.info(f"  Features used          : {len(vars_ok)} — WASI(1)")
logger.info(f"  K final                : {K_FINAL}")
logger.info(f"  Imputation             : MICE (BayesianRidge, 10 iter)")
logger.info(f"  Silhouette global      : {sil_global:.4f}")
logger.info(f"  Davies-Bouldin         : {db_score:.4f}")
for go in [1, 2]:
    n = (labels_go == go).sum()
    logger.info(f"  {LABEL_GO[go]:<14} n={n} ({n/len(labels_go)*100:.1f}%)")
logger.info(f"  MLflow run ID          : {run_id}")
logger.info("")
logger.info("  Key findings:")
logger.info("    WASI FSIQ: sole discriminator — K-Means equivalent to FSIQ threshold")
logger.info("    GO-1 High IQ: approximately 1 SD above GO-2 mean")
logger.info("    Compare with full GO-i: WASI alone tests whether IQ drives the composite phenotype")
logger.info("")
logger.info("  Next step:")
logger.info("    Use clusters_GO_WASI_v1.csv as alternative target for Part D")
